<a href="https://colab.research.google.com/github/hemasri159/Skill-Experiments_ML_2520030159/blob/main/EXP6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

file_path = '/content/drive/MyDrive/colab/temp.csv'

df = pd.read_csv(file_path)

print(df.head())
print("\nDataset Shape:", df.shape)

    1       2         3                            4       5    6       7  \
0  sn  pclass  survived                          NaN  gender  age  family   
1   1       3         0                  Mr. Anthony    male   42       0   
2   1       3         0                  Mr. Anthony    male   42       0   
3   2       3         0        Master. Eugene Joseph    male    ?       2   
4   3       2         0  Abbott, Mr. Rossmore Edward     NaN  NaN       2   

       8         9         10  
0   fare  embarked       date  
1   7.55       NaN  01-Jan-90  
2   7.55       NaN  01-Jan-90  
3  20.25         S  02-Jan-90  
4     **         S  03-Jan-90  

Dataset Shape: (1302, 10)


In [9]:
import os

folder_path = '/content/drive/MyDrive/colab'

print("Files in colabs:")
print(os.listdir(folder_path))

Files in colabs:
['placement_dataset.csv', 'placement_predict_50k Dataset (1).csv', 'placement_predict_50k_adjusted.csv', 'age_insurance.csv', 'e_commerce_user_data.csv.zip', 'simple_placement_dataset.csv', 'skillexp.csv', 'temp.csv']


In [12]:
file_path = '/content/drive/MyDrive/colab/temp.csv'

print("File exists:", os.path.exists(file_path))

File exists: True


In [13]:
df = pd.read_csv(file_path)

print(df.head())
print("\nDataset Shape:", df.shape)

    1       2         3                            4       5    6       7  \
0  sn  pclass  survived                          NaN  gender  age  family   
1   1       3         0                  Mr. Anthony    male   42       0   
2   1       3         0                  Mr. Anthony    male   42       0   
3   2       3         0        Master. Eugene Joseph    male    ?       2   
4   3       2         0  Abbott, Mr. Rossmore Edward     NaN  NaN       2   

       8         9         10  
0   fare  embarked       date  
1   7.55       NaN  01-Jan-90  
2   7.55       NaN  01-Jan-90  
3  20.25         S  02-Jan-90  
4     **         S  03-Jan-90  

Dataset Shape: (1302, 10)


In [14]:
print("Columns:")
print(df.columns)

print("\nDataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

Columns:
Index(['1', '2', '3', '4', '5', '6', '7', '8', '9', '10'], dtype='object')

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1302 entries, 0 to 1301
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   1       1302 non-null   object
 1   2       1302 non-null   object
 2   3       1302 non-null   object
 3   4       1301 non-null   object
 4   5       1301 non-null   object
 5   6       1045 non-null   object
 6   7       1300 non-null   object
 7   8       1300 non-null   object
 8   9       1296 non-null   object
 9   10      1302 non-null   object
dtypes: object(10)
memory usage: 101.8+ KB
None

Missing Values:
1       0
2       0
3       0
4       1
5       1
6     257
7       2
8       2
9       6
10      0
dtype: int64


In [16]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# The current 'df' DataFrame has numerical column names (1, 2, 3...)
# and the actual headers are in the first data row.
# Re-process 'df' to correctly set column names and clean the data.
processed_df = df.copy()

# Set column names using the first row of the DataFrame
processed_df.columns = processed_df.iloc[0]

# Drop the first row, as it's now the header
processed_df = processed_df[1:].reset_index(drop=True)

# The original header had a 'NaN' entry, which will now be a column name.
# Let's ensure all column names are strings and handle potentially problematic names.
new_columns = []
for col in processed_df.columns:
    if pd.isna(col):
        new_columns.append('unnamed_col') # Rename NaN column to 'unnamed_col'
    else:
        new_columns.append(str(col))
processed_df.columns = new_columns

# Convert relevant columns to numeric, handling problematic characters ('?', '**')
for col in ['age', 'fare', 'family']:
    if col in processed_df.columns:
        processed_df[col] = processed_df[col].replace({'?': np.nan, '**': np.nan})
        processed_df[col] = pd.to_numeric(processed_df[col], errors='coerce')


# Target variable
y = processed_df["pclass"] # Corrected column name from 'Pclass' to 'pclass'

# Input features
# Replaced 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'
# with actual column names: 'gender', 'age', 'family', 'fare', 'embarked'.
# 'family' is used as a proxy for 'SibSp' and 'Parch' as they are not separate columns.
X = processed_df[["gender", "age", "family", "fare", "embarked"]].copy() # Explicitly create a copy

# Categorical features
categorical_features = ["gender", "embarked"]

# Numerical features
numerical_features = ["age", "family", "fare"]

# Handle missing numerical values
for col in numerical_features:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce') # Ensure numeric type before median calculation
        X[col] = X[col].fillna(X[col].median())

# Handle missing categorical values
for col in categorical_features:
    if col in X.columns:
        X[col] = X[col].fillna(X[col].mode()[0])

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            drop="first"
        ), categorical_features)
    ]
)

# Create Logistic Regression pipeline
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]
)

# Train model
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Prediction probabilities
y_proba = model.predict_proba(X_test)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=[
        "First Class",
        "Second Class",
        "Third Class"
    ]
))

Accuracy: 0.8122605363984674

Confusion Matrix:
[[ 59   6   0]
 [  1  14  41]
 [  1   0 139]]

Classification Report:
              precision    recall  f1-score   support

 First Class       0.97      0.91      0.94        65
Second Class       0.70      0.25      0.37        56
 Third Class       0.77      0.99      0.87       140

    accuracy                           0.81       261
   macro avg       0.81      0.72      0.72       261
weighted avg       0.81      0.81      0.78       261



/tmp/ipykernel_4702/1458630197.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = pd.to_numeric(X[col], errors='coerce') # Ensure numeric type before median calculation
/tmp/ipykernel_4702/1458630197.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].median())
/tmp/ipykernel_4702/1458630197.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th